# Détection Textuelle — Baseline medical report

Notebook reproductible pour valider la brique texte du projet. Il utilise un petit CSV d'exemples cliniques et un baseline TF-IDF + régression logistique.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('data/sample/text_sample.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## Aperçu des données

On vérifie que les textes et les labels sont bien chargés avant l'entraînement.

In [ ]:
print(df.shape)
print(df['label'].value_counts().sort_index())
print(df['report_text'].iloc[0])

## Entraînement du baseline

On sépare un petit train/test set et on entraîne un modèle simple pour obtenir une validation rapide du pipeline texte.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['report_text'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=None)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
clf = LogisticRegression(max_iter=200)
clf.fit(X_train_vec, y_train)
pred = clf.predict(X_test_vec)
acc = accuracy_score(y_test, pred)
print(f'Accuracy test: {acc:.2f}')
print(classification_report(y_test, pred, digits=2))

## Matrice de confusion

Le baseline doit séparer les rapports cliniques positifs et négatifs sur ce petit jeu d'exemples.

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title('Matrice de confusion')
plt.tight_layout()
plt.show()

## Démo d'inférence

On teste une phrase clinique libre pour montrer le comportement du pipeline texte.

In [ ]:
sample_report = 'Toux persistante, fièvre et essoufflement depuis deux jours'
sample_vec = vectorizer.transform([sample_report])
sample_pred = clf.predict(sample_vec)[0]
sample_proba = clf.predict_proba(sample_vec)[0, 1]
print({'report': sample_report, 'prediction': int(sample_pred), 'probability': round(float(sample_proba), 3)})

## Conclusion

Ce notebook valide une baseline texte légère et exécutable localement. Il sert de support de test pour la modalité texte du projet.